## Coursera Dataset Cleaning

This notebook performs data cleaning on the Coursera.csv dataset. The cleaning steps include:
1. Reading and examining the data
2. Handling missing values
3. Removing duplicates
4. Standardizing text formats
5. Cleaning special characters
6. Processing Skills column
7. Saving the cleaned dataset

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re

# Read the dataset
df = pd.read_csv('../Data/raw/Coursera.csv')

In [2]:
# Examine the data
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

Dataset Shape: (3522, 7)

Columns: ['Course Name', 'University', 'Difficulty Level', 'Course Rating', 'Course URL', 'Course Description', 'Skills']

Missing Values:
 Course Name           0
University            0
Difficulty Level      0
Course Rating         0
Course URL            0
Course Description    0
Skills                0
dtype: int64

Data Types:
 Course Name           object
University            object
Difficulty Level      object
Course Rating         object
Course URL            object
Course Description    object
Skills                object
dtype: object


In [3]:
# 1. Clean special characters and standardize text
def clean_text(text):
    # Replace special characters and standardize text
    text = text.replace('�', "'")  # Replace smart quotes
    text = text.replace('�', '-')  # Replace em dash
    text = text.replace('�', 'e')  # Replace accented e
    text = text.replace('\u200b', '')  # Remove zero-width space
    text = re.sub(r'\s+', ' ', text)  # Remove multiple spaces
    return text.strip()

# Apply cleaning to text columns
text_columns = ['Course Name', 'University', 'Course Description', 'Skills']
for col in text_columns:
    df[col] = df[col].apply(clean_text)

# 2. Convert Course Rating to numeric
df['Course Rating'] = pd.to_numeric(df['Course Rating'].replace('Not Calibrated', np.nan))

# 3. Standardize Difficulty Level
df['Difficulty Level'] = df['Difficulty Level'].str.strip()
difficulty_mapping = {
    'Beginner': 'Beginner',
    'Intermediate': 'Intermediate',
    'Advanced': 'Advanced',
    'Not Calibrated': np.nan
}
df['Difficulty Level'] = df['Difficulty Level'].map(difficulty_mapping)

# 4. Process Skills column
def process_skills(skills):
    # Split skills by two or more spaces
    skills_list = re.split(r'\s{2,}', skills)
    # Clean individual skills
    skills_list = [skill.strip() for skill in skills_list]
    # Remove empty skills
    skills_list = [skill for skill in skills_list if skill]
    return ' | '.join(skills_list)

df['Skills'] = df['Skills'].apply(process_skills)

# 5. Remove duplicates
df.drop_duplicates(subset=['Course Name', 'University'], inplace=True)

# 6. Reset index
df.reset_index(drop=True, inplace=True)

In [4]:
# Examine the cleaned data
print("Final Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nDifficulty Level Values:\n", df['Difficulty Level'].value_counts(dropna=False))
print("\nCourse Rating Statistics:\n", df['Course Rating'].describe())

# Save the cleaned dataset
output_path = '../Data/processed/coursera_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\nCleaned dataset saved to: {output_path}")

Final Dataset Shape: (3424, 7)

Missing Values:
 Course Name             0
University              0
Difficulty Level      204
Course Rating          82
Course URL              0
Course Description      0
Skills                  0
dtype: int64

Difficulty Level Values:
 Difficulty Level
Beginner        1406
Advanced         991
Intermediate     823
NaN              204
Name: count, dtype: int64

Course Rating Statistics:
 count    3342.000000
mean        4.552244
std         0.340633
min         1.000000
25%         4.500000
50%         4.600000
75%         4.800000
max         5.000000
Name: Course Rating, dtype: float64

Cleaned dataset saved to: ../Data/processed/coursera_cleaned.csv


## Data Cleaning Summary

The following cleaning steps were performed on the Coursera dataset:

1. **Special Characters Cleaning**:
   - Replaced special quotes, dashes, and accented characters
   - Removed zero-width spaces
   - Standardized multiple spaces

2. **Course Rating**:
   - Converted to numeric format
   - Replaced 'Not Calibrated' with NaN
   - Final statistics show average rating of 4.55

3. **Difficulty Level Standardization**:
   - Mapped to three standard levels: Beginner, Intermediate, Advanced
   - Handled 'Not Calibrated' cases
   - Distribution: Beginner (1406), Advanced (991), Intermediate (823)

4. **Skills Processing**:
   - Split skills by multiple spaces
   - Cleaned individual skills
   - Joined with standard separator ' | '

5. **Data Quality**:
   - Removed duplicates based on Course Name and University
   - Reduced dataset from 3522 to 3424 records
   - Missing values identified in Difficulty Level (204) and Course Rating (82)

The cleaned dataset has been saved to: '../Data/processed/coursera_cleaned.csv'